In [271]:
import pandas as pd
from lifetimes import BetaGeoFitter, GammaGammaFitter

In [272]:
df = pd.read_parquet("../data/processed/cleaned_2021_customer_data.parquet")
df_2022 = pd.read_csv("../data/processed/cleaned_2022_customer_data.parquet")

In [274]:
df_2021 = df[["customer_id", "first_purchase_date", "last_purchase_date", "customer_tenure",
               "purchase_recency", "transaction_count", "total_spent", "avg_order_value"]].copy()
df_2021

,customer_id,first_purchase_date,last_purchase_date,customer_tenure,purchase_recency,transaction_count,total_spent,avg_order_value
0,21644,2021-01-01 01:14:00,2021-01-01 01:14:00,364.948454,364.948454,1,58.543,58.543
1,73192,2021-01-01 09:21:00,2021-01-01 09:21:00,364.610004,364.610004,1,74.030,74.030
2,9779,2021-01-01 16:04:00,2021-01-01 16:04:00,364.330234,364.330234,1,4.094,4.094
3,98867,2021-01-02 01:37:00,2021-01-02 01:37:00,363.932382,363.932382,1,14.923,14.923
4,48935,2021-01-02 09:29:00,2021-01-02 09:29:00,363.604602,363.604602,1,35.029,35.029
...,...,...,...,...,...,...,...,...
36081,9997,2021-01-05 16:20:00,2021-12-31 23:53:00,360.318883,0.004461,39,7082.052,181.591
36082,30112,2021-09-20 23:06:00,2021-12-31 23:54:00,102.036880,0.004067,17,2084.390,122.611
36083,7386,2021-01-02 21:04:00,2021-12-31 23:54:00,363.121734,0.003747,118,13513.165,114.518
36084,56269,2021-08-16 01:24:00,2021-12-31 23:56:00,137.941532,0.002516,20,502.177,25.109


<br/>

In [275]:
df_2022

,Unnamed: 0,customer_id,first_purchase_date,last_purchase_date,total_spent,transaction_count,avg_order_value
0,0,68449,2022-01-01 00:20:00,2022-01-01 00:20:00,19.039,1,19.039
1,1,78745,2022-01-01 00:32:00,2022-01-01 00:32:00,43.349,1,43.349
2,2,37989,2022-01-01 01:27:00,2022-01-01 01:27:00,17.070,1,17.070
3,3,16051,2022-01-01 01:42:00,2022-01-01 01:42:00,15.703,1,15.703
4,4,74599,2022-01-01 01:49:00,2022-01-01 01:49:00,22.640,1,22.640
...,...,...,...,...,...,...,...
32418,32418,15753,2022-07-31 23:44:00,2022-07-31 23:58:00,139.978,8,17.497
32419,32419,27522,2022-07-31 23:31:00,2022-07-31 23:58:00,46.415,2,23.207
32420,32420,81975,2022-07-31 07:02:00,2022-07-31 23:58:00,1450.199,41,35.371
32421,32421,22285,2022-07-31 23:59:00,2022-07-31 23:59:00,9.253,1,9.253


In [276]:
# Frequency = customers with repeat purchases (more than 1)
df_2021['btyd_frequency'] = df_2021['transaction_count'] - 1  
# btyd models only account for customers who made more than one transaction


# Recency = Days between first and last purchase
df_2021['btyd_recency'] = (pd.to_datetime(df_2021['last_purchase_date']) - pd.to_datetime(df_2021['first_purchase_date'])).dt.days


# T (Tenure) = Days between first purchase and the end of your observation period
max_date = pd.to_datetime(df_2021['last_purchase_date']).max()
df_2021['btyd_T'] = (max_date - pd.to_datetime(df_2021['first_purchase_date'])).dt.days


# BTYD Monetary = Average order value
df_2021['btyd_monetary'] = df_2021['avg_order_value']

In [277]:
# Filtering out one-time buyers for the monetary model
# Gamma-Gamma model only works on customers who have repeat purchases
repeat_buyers = df_2021[df_2021['btyd_frequency'] > 0].copy()

### BetaGeoModel Fitter

In [278]:
bgf = BetaGeoFitter(penalizer_coef=0.0)
# Takes in frequency, recency and tenure
bgf.fit(df_2021['btyd_frequency'], df_2021['btyd_recency'], df_2021['btyd_T'])

c:\Users\NewAdminUser\OneDrive\Documents\DANNY DATA\Ecommerce CLV Prediction\.venv\lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


<lifetimes.BetaGeoFitter: fitted with 36086 subjects, a: 0.00, alpha: 32.34, b: 8810.85, r: 0.66>

<br/>

In [279]:
# Probability that a particular customer is still actively with us
df_2021['prob_alive'] = bgf.conditional_probability_alive(
    df_2021['btyd_frequency'], df_2021['btyd_recency'], df_2021['btyd_T']
)



# Predicting how man purchases they would make in the next 90 days
t = 90
df_2021['predicted_purchases_90d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    t, df_2021['btyd_frequency'], df_2021['btyd_recency'], df_2021['btyd_T']
)


In [286]:
df_repeat.head()

,customer_id,first_purchase_date,last_purchase_date,customer_tenure,purchase_recency,transaction_count,total_spent,avg_order_value,btyd_frequency,btyd_recency,btyd_T,btyd_monetary,expected_avg_spend,predicted_clv_3m
1824,82104,2021-01-10 11:00:00,2021-04-14 13:23:00,355.541500,261.441952,2,56.614,28.307,1,94,355,28.307,35.099410,13.233513
2116,57056,2021-01-27 16:29:00,2021-04-30 17:59:00,338.312887,245.250642,2,286.532,143.266,1,93,338,143.266,141.287851,55.714951
2149,73701,2021-01-31 02:58:00,2021-05-02 03:30:00,334.875738,243.854014,2,35.365,17.683,1,91,334,17.683,25.285946,10.080043
2219,96781,2021-01-11 22:09:00,2021-05-04 22:08:00,354.076960,241.077376,2,21.681,10.841,1,112,354,10.841,18.965942,7.169228
2439,78400,2021-02-16 22:53:00,2021-05-16 21:47:00,318.045874,229.091950,2,3234.170,1617.085,1,88,318,1617.085,1502.664886,626.382773


<br/>

### Gamma-Gamma Model to predict customer value

In [282]:
print(df_repeat[['btyd_frequency', 'btyd_monetary']].corr())

                btyd_frequency  btyd_monetary
btyd_frequency        1.000000      -0.007102
btyd_monetary        -0.007102       1.000000


In [283]:
ggf = GammaGammaFitter(penalizer_coef=0)
ggf.fit(df_repeat['btyd_frequency'], df_repeat['btyd_monetary'])

<lifetimes.GammaGammaFitter: fitted with 26707 subjects, p: 5.09, q: 1.42, v: 9.69>

In [284]:
# # Predict expected average transaction value for the future
df_repeat['expected_avg_spend'] = ggf.conditional_expected_average_profit(
    df_repeat['btyd_frequency'], df_repeat['btyd_monetary']
)

In [285]:
df_repeat.head()

,customer_id,first_purchase_date,last_purchase_date,customer_tenure,purchase_recency,transaction_count,total_spent,avg_order_value,btyd_frequency,btyd_recency,btyd_T,btyd_monetary,expected_avg_spend,predicted_clv_3m
1824,82104,2021-01-10 11:00:00,2021-04-14 13:23:00,355.541500,261.441952,2,56.614,28.307,1,94,355,28.307,35.099410,13.233513
2116,57056,2021-01-27 16:29:00,2021-04-30 17:59:00,338.312887,245.250642,2,286.532,143.266,1,93,338,143.266,141.287851,55.714951
2149,73701,2021-01-31 02:58:00,2021-05-02 03:30:00,334.875738,243.854014,2,35.365,17.683,1,91,334,17.683,25.285946,10.080043
2219,96781,2021-01-11 22:09:00,2021-05-04 22:08:00,354.076960,241.077376,2,21.681,10.841,1,112,354,10.841,18.965942,7.169228
2439,78400,2021-02-16 22:53:00,2021-05-16 21:47:00,318.045874,229.091950,2,3234.170,1617.085,1,88,318,1617.085,1502.664886,626.382773
